In [45]:
import pandas as pd
import re

### Dataset info
| show_id | type | title | director | cast | country | date_added | release_year | rating | duration | listed_in | description |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| 8807 unique values| Movie 70%<br> TvShow 30% | 8807 unique values | [null] 30%<br>Other 70% | [null] 9%<br> Other 90% | country of prodcution | date it was added on netflix | Actual release year of the movie/show | Tv-rating | Total duration in minutes or number of seasons | Genre<br> (Dramas, Documentaries, etc) | The summary description |

In [46]:
# Read the data
movies = pd.read_csv("../data/netflix_titles.csv")

# We keep just the first 1000 samples
movies = movies[:1000]

print(f"Number of samples: {len(movies)}")

Number of samples: 1000


In [47]:
import unicodedata

def clean_string(val):
    if not isinstance(val, str):
        return val
    val = unicodedata.normalize('NFKD', val)
    val = val.encode('ascii', errors='ignore').decode('ascii')
    val = val.replace('&', 'and')  # replace & with 'and'
    val = val.strip()
    return val

movies = movies.map(clean_string)

In [48]:
movies.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood and Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [49]:
# Auxilary function to count the number of occurrences 
# of the values for a given attribute

def counter(attr):
    count = {}

    if attr not in movies.columns:
        return "Error: attribute not present"
    
    col = movies[attr]

    for elem in col:
        if elem in count:
            count[elem] += 1
        else:
            count[elem] = 1

    return count

In [50]:
count = counter('rating')
print(count)

{'PG-13': 101, 'TV-MA': 339, 'PG': 44, 'TV-14': 211, 'TV-PG': 64, 'TV-Y': 35, 'TV-Y7': 59, 'R': 120, 'TV-G': 25, 'G': 2}


### Cassandra CQL files

In the following we prepare the cql files to create and populate the database.

#### Database *"schema"*

As A first step we need to create the **KEYSPACE**:

```cql
CREATE KEYSPACE IF NOT EXISTS movies
    WITH REPLICATION = {"class": "SimpleStrategy","replication_factor": 1};
```

Then we define the table (or in other words the **column families**):

``` cql
CREATE TABLE IF NOT EXISTS movies.Netflix_titles (  
    show_id text,
    type text, 
    title text, 
    rating text,   
    duration text, 
    director text, 
    cast set<text>,     ← NOTE: Cassandra is able to efficiently handle also collections
    release_year int,
    country text, 
    PRIMARY KEY (type, show_id)  
) 
```

We are also going to define a partition based on the ratings.
`{'PG-13': 101, 'TV-MA': 339, 'PG': 44, 'TV-14': 211, 'TV-PG': 64, 'TV-Y': 35, 'TV-Y7': 59, 'R': 120, 'TV-G': 25, 'G': 2}`

```cql
CREATE TABLE IF NOT EXISTS movies.ratings (
    show_id text, 
    rating text, 
    type text, 
    title text, 
    duration text, 
    description text, 
    PRIMARY KEY (rating, type, show_id)
)
```


In [ ]:
def generate_cql_files(data, attributes, table, path):
    ''' 
        data: DataFrame
        columns: the columns of the column family that we wan to populate
        table: the name of the column family
        path: path of the .cql file where to write the data
    '''

    num_rows = len(data)

    with open(path, 'w', encoding='utf-8') as f:

        for idx in range(num_rows):

            row = data.iloc[idx]

            valid_attrs = [attr for attr in attributes if not pd.isna(row[attr])] # select only the columns with not NaN values
            columns = ", ".join(valid_attrs)    
            vals_list = []
            for col in valid_attrs:

                val= row[col]

                # if isinstance(val, str):
                #     val = re.sub(r'\s', ' ', val)

                if col == 'cast':
                    ### Save the cast as a set
                    actors_list = [a.strip() for a in row[col].split(',')]
                    formatted_actors = [f"'{a.replace(chr(39), chr(39)*2)}'" for a in actors_list] 
                    set_str = "{" + ", ".join(formatted_actors) + "}"
                    vals_list.append(set_str)
                    
                elif isinstance(val, str):
                    val = val.replace(chr(39), chr(39)*2)
                    vals_list.append(f"'{val}'")
                
                else:
                    vals_list.append(str(val))

            values = ", ".join(vals_list)

            # genearte the final command to insert the values in the database
            line = f"INSERT INTO {table} ({columns})\nVALUES ({values});\n"
            
            f.write(line)

    
    return f"File for table {table} ready"


In [52]:
netflix_titles = ['show_id', 'type', 'title', 'rating', 'duration', 'director', 'cast', 'release_year', 'country']
ratings = ['show_id', 'rating', 'type', 'title', 'duration', 'description']

path1 = '../cql-scripts/netfilx_titles.cql'
path2 = '../cql-scripts/ratings.cql'

generate_cql_files(data=movies, attributes=netflix_titles, table='movies.Netflix_titles', path=path1)
generate_cql_files(data=movies, attributes=ratings, table='movies.ratings', path=path2)

'File for table movies.ratings ready'

In [ ]:
row = movies.iloc[5]
# print(row)

#print(isinstance(row['cast'], str))
vals = []

actors_list = [a.strip() for a in row['cast'].split(',')]

# f"'{a.strip().replace("'", "''")}'" for a in actors_list

formatted_actors = [f"'{a}'" for a in actors_list]
set_str = "{" + ", ".join(formatted_actors) + "}"
print(type(set_str))

vals.append(set_str)

#print(vals)

<class 'str'>


In [19]:
row = movies.iloc[149]
print(row['cast'])

actors = [a.strip() for a in row['cast'].split(',')]
formatted_actors = [f"'{a.replace(chr(39), chr(39)*2)}'" for a in actors]
set_str = "{" + ", ".join(formatted_actors) + "}"
print(set_str)

cleaned = re.sub(r"(?<=\w)'(?=\w)", "''", set_str)
print(cleaned)


Master P, Anthony Johnson, Gretchen Palmer, Frantz Turner, Richard Keats, Joe Estevez, William Knight, Anthony Boswell, Tommy 'Tiny' Lister, Helen Martin, John Witherspoon, Mia X
{'Master P', 'Anthony Johnson', 'Gretchen Palmer', 'Frantz Turner', 'Richard Keats', 'Joe Estevez', 'William Knight', 'Anthony Boswell', 'Tommy ''Tiny'' Lister', 'Helen Martin', 'John Witherspoon', 'Mia X'}
{'Master P', 'Anthony Johnson', 'Gretchen Palmer', 'Frantz Turner', 'Richard Keats', 'Joe Estevez', 'William Knight', 'Anthony Boswell', 'Tommy ''Tiny'' Lister', 'Helen Martin', 'John Witherspoon', 'Mia X'}


In [18]:
ex = "bla'"
new = ex.replace('\'', '')
print(new)
print(chr(39))

bla
'


In [21]:
print(vals)

["{{' Alex Essoe', ' Rahul Kohli', ' Michael Trucco', ' Zach Gilford', ' Annabeth Gish', 'Kate Siegel', ' Hamish Linklater', ' Henry Thomas', ' Kristin Lehman', ' Igby Rigney', ' Crystal Balint', ' Samantha Sloyan', ' Rahul Abburi', ' Matt Biedel', ' Louis Oliver', ' Annarah Cymone'}}"]


In [8]:
rows = len(movies)

for idx in range(rows): 
    row = movies.iloc[idx]
    print(row['type'])
    #print(movies.iloc[idx])
    # print(len(movies.iloc[idx]))
    break

#print(movies.shape)

Movie


In [30]:
with open('../cql-scripts/netfilx_titles.cql', 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Line 1245 in the error (0-indexed = 1244)
line1 = lines[1244]
print(repr(line1))
line2 = lines[1245]
print(repr(line2))

'INSERT INTO movies.Netflix_titles (show_id, type, title, rating, duration, director, cast, release_year, country)\n'
"VALUES ('s623', 'Movie', 'Lying and Stealing', 'R', '100 min', 'Matt Aselton', {'Theo James', 'Emily Ratajkowski', 'Fred Melamed', 'Ebon Moss-Bachrach', 'Isiah Whitlock Jr.', 'Evan Handler', 'Paul Jurewicz', 'John Gatins', 'Fernanda Andrade', 'Bob Stephenson'}, 2019, 'United States');\n"


In [31]:
with open('../cql-scripts/netfilx_titles.cql', 'r', encoding='utf-8') as f:
    content = f.read()

statements = content.split(';\n')

for i, stmt in enumerate(statements):
    if 's622' in stmt:
        print(f"Statement index: {i}")
        print(repr(stmt))
        

Statement index: 621
"INSERT INTO movies.Netflix_titles (show_id, type, title, rating, duration, director, cast, release_year)\nVALUES ('s622', 'TV Show', 'Legend\xa0of\xa0Exorcism', 'TV-14', '1 Season', 'Shen Leping', {'Bian Jiang', 'Chen Jinwen', 'Ling Fei'}, 2020)"
